# Epic on FHIR Auth Model

Notebook for Epic on FHIR auth model workflows.

In [1]:
import os

In [2]:
# Initialize Databricks Connect when running locally (spark/dbutils are pre-injected on Databricks)
try:
    spark
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().profile("fe-vm-mkgs-databricks-demos").getOrCreate()
    os.environ["EPIC_ON_FHIR_MLFLOW_USE_PROFILE"] = "1"

In [3]:
# Widget definitions (defaults from bundle variables)
# catalog_use, schema_use, secret_scope_name, client_id_dbs_key, algo, token_url,
# mlflow_experiment_name (deployed experiment), registered_model_name (deployed model)

_catalog_use = "mkgs_dev"
_schema_use = "dev_matthew_giglia_epic_on_fhir"
_secret_scope_name = "epic_on_fhir_oauth_keys"
_client_id_dbs_key = "client_id"
_algo = "RS384"
_token_url = "https://fhir.epic.com/interconnect-fhir-oauth/oauth2/token"
_mlflow_experiment_name = "/Workspace/experiments/[dev matthew_giglia] epic_on_fhir_auth"
_registered_model_name = "mkgs_dev.dev_matthew_giglia_epic_on_fhir.epic_on_fhir_auth"

try:
    dbutils.widgets.text("catalog_use", _catalog_use, "Catalog")
    dbutils.widgets.text("schema_use", _schema_use, "Schema")
    dbutils.widgets.text("secret_scope_name", _secret_scope_name, "Epic Secrets Scope")
    dbutils.widgets.text("client_id_dbs_key", _client_id_dbs_key, "Epic Client ID DB Secrets Key")
    dbutils.widgets.text("algo", _algo, "Epic Token Encryption Algorithm")
    dbutils.widgets.text("token_url", _token_url, "Epic Token URL")
    dbutils.widgets.text("mlflow_experiment_name", _mlflow_experiment_name, "MLflow Experiment Name")
    dbutils.widgets.text("registered_model_name", _registered_model_name, "Registered Model Name")
except Exception:
    pass  # Databricks Connect: widget creation may not be fully supported

Box(children=(Label(value='Catalog'), Text(value='mkgs_dev')))

Box(children=(Label(value='Schema'), Text(value='dev_matthew_giglia_epic_on_fhir')))

Box(children=(Label(value='Epic Secrets Scope'), Text(value='epic_on_fhir_oauth_keys')))

Box(children=(Label(value='Epic Client ID DB Secrets Key'), Text(value='client_id')))

Box(children=(Label(value='Epic Token Encryption Algorithm'), Text(value='RS384')))

Box(children=(Label(value='Epic Token URL'), Text(value='https://fhir.epic.com/interconnect-fhir-oauth/oauth2/…

Box(children=(Label(value='MLflow Experiment Name'), Text(value='/Workspace/experiments/[dev matthew_giglia] e…

Box(children=(Label(value='Registered Model Name'), Text(value='mkgs_dev.dev_matthew_giglia_epic_on_fhir.epic_…

In [ ]:
# Retrieve widget values (fallback for Databricks Connect)
try:
    catalog_use = dbutils.widgets.get("catalog_use")
    schema_use = dbutils.widgets.get("schema_use")
    secret_scope_name = dbutils.widgets.get("secret_scope_name")
    client_id_dbs_key = dbutils.widgets.get("client_id_dbs_key")
    algo = dbutils.widgets.get("algo")
    token_url = dbutils.widgets.get("token_url")
    mlflow_experiment_name = dbutils.widgets.get("mlflow_experiment_name")
    registered_model_name = dbutils.widgets.get("registered_model_name")
except (AttributeError, Exception):
    catalog_use = _catalog_use
    schema_use = _schema_use
    secret_scope_name = _secret_scope_name
    client_id_dbs_key = _client_id_dbs_key
    algo = _algo
    token_url = _token_url
    mlflow_experiment_name = _mlflow_experiment_name
    registered_model_name = _registered_model_name

In [ ]:
import sys
from pathlib import Path
# Ensure epic_on_fhir package is importable
_root = Path().resolve()
if _root.name == "src":
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import pandas as pd
import mlflow
from epic_on_fhir.epic_fhir_pyfunc import EpicFhirPyfuncModel

In [ ]:
# Only set explicit MLflow URIs when running locally (Databricks Connect).
# On Databricks (notebook/job on cluster or serverless), MLflow is preconfigured.
# Use opt-in env var: set EPIC_ON_FHIR_MLFLOW_USE_PROFILE=1 when running locally.
# DATABRICKS_RUNTIME_VERSION is not reliably set on serverless.

if os.environ.get("EPIC_ON_FHIR_MLFLOW_USE_PROFILE", "").strip() in ("1", "true", "yes"):
    _mlflow_profile = "fe-vm-mkgs-databricks-demos"
    mlflow.set_tracking_uri(f"databricks://{_mlflow_profile}")
    mlflow.set_registry_uri(f"databricks-uc://{_mlflow_profile}")

In [ ]:
# Use experiment and registered model from widget defaults.
# The experiment is deployed by the bundle with artifact_location in the mlflow_artifacts volume.
# set_experiment switches to it; if it doesn't exist, creates with default location.
mlflow.set_experiment(mlflow_experiment_name)

<Experiment: artifact_location='dbfs:/Volumes/mkgs_dev/dev_matthew_giglia_epic_on_fhir/mlflow_artifacts', creation_time=1771664943854, experiment_id='2431848039072739', last_update_time=1771664943854, lifecycle_stage='active', name='/experiments/[dev matthew_giglia] epic_on_fhir_auth', tags={'description': 'MLflow experiment for Epic on FHIR Authentication workflows '
                'and model runs',
 'dev': 'matthew_giglia',
 'mlflow.experiment.sourceName': '/experiments/[dev matthew_giglia] '
                                 'epic_on_fhir_auth',
 'mlflow.experimentType': 'MLFLOW_EXPERIMENT',
 'mlflow.ownerEmail': 'matthew.giglia@databricks.com',
 'mlflow.ownerId': '1081964970114387'}, workspace='default'>

In [ ]:
# Build the pyfunc model with widget values
model = EpicFhirPyfuncModel(
    secret_scope_name=secret_scope_name,
    client_id_dbs_key=client_id_dbs_key,
    token_url=token_url,
    algo=algo,
)

In [ ]:
import json
from pathlib import Path

In [21]:
# Input example for signature: http_method, resource, action, data (optional)
# Row 1: Patient GET | Row 2: Observation create | Row 3: AllergyIntolerance create

import random
from datetime import date, timedelta

# Keep date 2019-09-05, randomly select a valid time for effectiveDateTime
_obs_date = "2019-09-05"
_obs_time = f"{random.randint(0, 23):02d}:{random.randint(0, 59):02d}:{random.randint(0, 59):02d}"
_effective_dt = f"{_obs_date}T{_obs_time}Z"

# Random date in 2024 for allergy recordedDate (2024 is leap year, 366 days)
_recorded_date = (date(2024, 1, 1) + timedelta(days=random.randint(0, 365))).isoformat()

observation_payload = {
    "resourceType": "Observation",
    "status": "final",
    "category": [{"coding": [{"system": "http://hl7.org/fhir/observation-category", "code": "vital-signs", "display": "Vital Signs"}], "text": "Vital Signs"}],
    "code": {"coding": [{"system": "urn:oid:1.2.840.114350.1.13.0.1.7.2.707679", "code": "8", "display": "Heart Rate"}], "text": "Heart Rate"},
    "subject": {"reference": "Patient/T1wI5bk8n1YVgvWk9D05BmRV0Pi3ECImNSK8DKyKltsMB"},
    "encounter": {"reference": "Encounter/e0u1fd.jUCNqz8ZQuTaMtsQ3"},
    "effectiveDateTime": _effective_dt,
    "valueQuantity": {"value": 75},
}

allergy_payload = {
    "resourceType": "AllergyIntolerance",
    "clinicalStatus": {"coding": [{"system": "http://terminology.hl7.org/CodeSystem/allergyintolerance-clinical", "code": "active", "display": "Active"}], "text": "Active"},
    "verificationStatus": {"coding": [{"system": "http://terminology.hl7.org/CodeSystem/allergyintolerance-verification", "code": "unconfirmed", "display": "Unconfirmed"}], "text": "Unconfirmed"},
    "type": "allergy",
    "category": ["medication"],
    "criticality": "low",
    "code": {"coding": [{"system": "http://www.nlm.nih.gov/research/umls/rxnorm", "code": "7980", "display": "Penicillin G"}], "text": "Penicillin"},
    "patient": {"reference": "Patient/T1wI5bk8n1YVgvWk9D05BmRV0Pi3ECImNSK8DKyKltsMB"},
    "recorder": {"reference": "Practitioner/eM5CWtq15N0WJeuCet5bJlQ3", "display": "Physician Family Medicine, MD"},
    "recordedDate": _recorded_date,
    "reaction": [{"manifestation": [{"coding": [{"system": "http://snomed.info/sct", "code": "247472004", "display": "Hives"}], "text": "Hives"}]}],
}

input_example = pd.DataFrame([
    {"http_method": "get", "resource": "Patient", "action": "T1wI5bk8n1YVgvWk9D05BmRV0Pi3ECImNSK8DKyKltsMB"},
    {"http_method": "post", "resource": "Observation", "action": "", "data": json.dumps(observation_payload)},
    {"http_method": "post", "resource": "AllergyIntolerance", "action": "", "data": json.dumps(allergy_payload)},
])

In [22]:
# Conda env for model serving: JWT (PyJWT), cryptography (for RS384), requests, pandas, mlflow
_conda_env = {
    "name": "epic_on_fhir_serving",
    "channels": ["conda-forge"],
    "dependencies": [
        "python=3.12",
        "pip",
        {"pip": ["PyJWT", "cryptography", "requests", "pandas", "mlflow>=3.1"]},
    ],
}

In [23]:
# Include epic_on_fhir package source for serving (not pip-installable in serving env)
_src_path = Path(_root) / "src"
_code_path = [str(_src_path)] if _src_path.exists() else []

# Infer MLflow signature from model_input (optionally run model.predict(input_example) for output schema)
from mlflow.models import infer_signature

try:
    _model_output = model.predict(input_example)
    _signature = infer_signature(model_input=input_example, model_output=_model_output)
except Exception:
    _signature = infer_signature(model_input=input_example)

In [ ]:

# Log model to MLflow 3 and register (no start_run needed; models are first-class entities)
model_info = mlflow.pyfunc.log_model(
    name="epic_on_fhir_requests",
    python_model=model,
    registered_model_name=registered_model_name,
    input_example=input_example,
    signature=_signature,
    conda_env=_conda_env,
    code_paths=_code_path if _code_path else None,
    params={
        "secret_scope_name": secret_scope_name,
        "client_id_dbs_key": client_id_dbs_key,
        "token_url": token_url,
        "algo": algo,
    },
)
model_info  # display model_uri, model_id

🔗 View Logged Model at: https://fe-vm-mkgs-databricks-demos.cloud.databricks.com/ml/experiments/2431848039072739/models/m-c23a6c4e69634230a4d7c3458101bed7
2026/02/21 04:20:58 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.
2026/02/21 04:20:58 INFO mlflow.pyfunc: Validating input example against model signature
2026/02/21 04:20:58 WARNING mlflow.models.model: Failed to validate serving input example {
  "dataframe_split": {
    "columns": [
      "h.... Alternatively, you can avoid passing input example and pass model signature instead when logging the model. To ensure the input example is valid prior to serving, please try calling `mlflow.models.validate_serving_input` on the mode

MlflowException: Reading Databricks credential configuration failed with MLflow registry URI 'databricks-uc'. Please ensure that the 'databricks-sdk' PyPI library is installed, the tracking URI is set correctly, and Databricks authentication is properly configured. The registry URI can be either 'databricks-uc' (using profile name specified by 'DATABRICKS_CONFIG_PROFILE' environment variable or using 'DEFAULT' authentication profile if 'DATABRICKS_CONFIG_PROFILE' environment variable does not exist) or 'databricks-uc://{profile}'. You can configure Databricks authentication in several ways, for example by specifying environment variables (e.g. DATABRICKS_HOST + DATABRICKS_TOKEN) or logging in using 'databricks auth login'. 
For details on configuring Databricks authentication, please refer to 'https://docs.databricks.com/en/dev-tools/auth/index.html#unified-auth'.

In [ ]:
# Optional: test prediction (requires Epic secrets in scope). Uncomment to run.
# In MLflow 3, use model_uri from log_model result or models:/name/version
# Same example as input_example: http_method, resource, action
# test_df = pd.DataFrame([{"http_method": "get", "resource": "Patient", "action": "T1wI5bk8n1YVgvWk9D05BmRV0Pi3ECImNSK8DKyKltsMB"}])
# loaded = mlflow.pyfunc.load_model(model_info.model_uri)  # from cell above
# # or: mlflow.pyfunc.load_model(f"models:/{registered_model_name}/1")
# loaded.predict(test_df)